In [1]:
import geopandas as gpd
import libpysal as lp
import pandas as pd
import os
import pyarrow as pa
import pyarrow.parquet as pq
import numpy as np

In [ ]:
#CHECK PATH

map_2023_path = "../../housing-data/cbsgebiedsindelingen2023.gpkg"

In [ ]:
def read_file(file_path, layer="gemeente_gegeneraliseerd"):
    gdf = gpd.read_file(file_path, layer=layer)
    #gdf = gdf[gdf['indelingswijziging_wijken_en_buurten'] > 0] -- Wel of niet???
    gdf_neighbors = lp.weights.Queen.from_dataframe(gdf, use_index=False)
    gdf_neighbors.to_sparse()
    codes = gdf.iloc[:, :1].to_numpy().flatten()
    adj_df = pd.DataFrame.sparse.from_spmatrix(
        gdf_neighbors.to_sparse(), index=codes, columns=codes
    )
    return adj_df, gdf

adj_df_2023, gdf_2023 = read_file(map_2023_path)

In [ ]:
adj_df_2023.to_csv("../../housing-data/buurt_adj_2023.csv")

In [ ]:
def read_file(file_path, layer="buurt_gegeneraliseerd"):
    gdf = gpd.read_file(file_path, layer=layer)
    #gdf = gdf[gdf['indelingswijziging_wijken_en_buurten'] > 0] -- Wel of niet???
    gdf_neighbors = lp.weights.Queen.from_dataframe(gdf, use_index=False)
    gdf_neighbors.to_sparse()
    codes = gdf.iloc[:, :1].to_numpy().flatten()
    adj_df = pd.DataFrame.sparse.from_spmatrix(
        gdf_neighbors.to_sparse(), index=codes, columns=codes
    )
    return adj_df, gdf

adj_df_2023, gdf_2023 = read_file(map_2023_path)

In [ ]:
adj_2023 = pd.read_csv("../../housing-data/buurt_adj_2023.csv",index_col=0)

In [ ]:
gemeente_rotterdam = ['GM1930',
 'GM0502',
 'GM0622',
 'GM0597',
 'GM0556',
 'GM0489',
 'GM0606',
 'GM1621',
 'GM1992',
 'GM0613',
 'GM0542',
 'GM0599']

In [ ]:
buurten_rotterdam = list(gdf_2023[gdf_2023["gm_code"].isin(gemeente_rotterdam)]["statcode"].values)
print(buurten_rotterdam)

In [ ]:
rotterdam_adj_2023 = adj_2023.loc[buurten_rotterdam, buurten_rotterdam]
rotterdam_adj_2023

In [ ]:
all_transactions = pd.read_csv("../../housing-data/transaction_data.csv")

In [ ]:
index_to_code = dict(enumerate(adj_2023.columns))

# Replace the buurtcode column with actual codes
all_transactions["BUURTCODE"] = all_transactions["BUURTCODE"].map(index_to_code)

In [ ]:
adj_rot = pd.read_csv("../../housing-data/rotterdam_adj_2023.csv", index_col=0)

In [ ]:
assert(adj_2023.columns.equals(rotterdam_adj_2023.columns))

In [ ]:
assert(set(adj1.columns) == set(adj2.columns))

In [ ]:

transactions_rot = all_transactions[all_transactions["BUURTCODE"].isin(buurten_rotterdam)].copy()


In [ ]:
buurten = adj_rot.columns.values
label_encoder = LabelEncoder()
label_encoder.fit(buurten)

transactions_rot["BUURTCODE"] = label_encoder.transform(transactions_rot["BUURTCODE"])

In [ ]:

print(transactions_rot)

In [ ]:
# Check index = False part based on structure of above output
transactions_rot.to_csv("../../housing-data/rotterdam_transaction_data.csv", index=False)